In [ ]:
import os, gc, time, copy, wandb, random, logging

from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import polars as pl

from dotenv import load_dotenv

import torch

from sklearn.model_selection import GroupKFold

from umap import UMAP
from umap.utils import disconnected_vertices

from hdbscan import HDBSCAN

from sentence_transformers import SentenceTransformer

from sentence_transformers.cross_encoder.training_args import CrossEncoderTrainingArguments
from sentence_transformers.cross_encoder import CrossEncoder, CrossEncoderTrainer
from sentence_transformers.cross_encoder.losses import ADRMSELoss, LambdaLoss

from transformers import BitsAndBytesConfig, TrainerCallback

from datasets import Dataset

from peft import LoraConfig, TaskType, prepare_model_for_kbit_training
from peft.utils import get_peft_model_state_dict, set_peft_model_state_dict

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

In [ ]:
# Load datasets, encode label as `label_id`, and add context to dataframes
TRAIN_PATH = "../../data/train.csv"
TEST_PATH = "../../data/test.csv"

OPTION_COLS = ["A", "B", "C", "D", "E"]

LABEL_COL = "answer"
LABEL2ID = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}
ID2LABEL = {0: "A", 1: "B", 2: "C", 3: "D", 4: "E"}

train_data = pl.read_csv(TRAIN_PATH)
test_data = pl.read_csv(TEST_PATH)

train_data = train_data.with_columns(
    pl.col(LABEL_COL).replace(LABEL2ID).cast(pl.Int8).alias("label_id")
)

train_data = train_data.with_columns(
    (
        pl.lit("Prompt : ") + pl.col("prompt")
        + pl.lit(" Options: ")
        + pl.concat_str(
            [pl.lit(f"{opt}) ") + pl.col(opt).fill_null(" ") for opt in OPTION_COLS],
            separator=" "
        )
    ).alias("mcq_query")
)

ids = train_data["id"].to_list()

In [ ]:
# Configure training parameters, data, models, device, and seeds for reproducibility
EPOCHS = 3
N_SPLITS = 5

PER_DEVICE_BATCH_SIZE = 8
GRAD_ACCUMULATION_STEPS = 8

ID_COL = "id"
QUESTION_COL = "prompt"
OPTION_COLS  = ["A", "B", "C", "D", "E"]

EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
RANK_MODEL = "zeroentropy/zerank-2-reranker"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 42

def set_seed(seed: int, is_final_run: bool = False) -> None:
    
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False if not is_final_run else True
    torch.backends.cudnn.deterministic = True if not is_final_run else False

set_seed(SEED, is_final_run = False)

In [ ]:
# Configure quantization config, model instructions, best model directory, loggers, and API keys
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

qlora_bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

MCQ_RERANK_INSTRUCTION = (
    "Given a multiple-choice question, determine whether the candidate answer is the "
    "single factually correct answer to that question. Score it high only if it is "
    "precise, accurate, and directly resolves what the question is asking. Score it "
    "low if it is a plausible-sounding distractor, a partially correct answer, an "
    "answer that is topically related but does not actually answer the question, or "
    "one that is factually wrong. The questions span a wide range of domains "
    "physics, chemistry, biology, astronomy, mathematics, philosophy, and other "
    "STEM and humanities topics - so judge correctness using domain-appropriate "
    "reasoning, not surface-level keyword overlap between the question and the answer."
)

def reset_gpu() -> None:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Configure logging levels to hide model-loading report
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.WARNING)
logging.getLogger("huggingface_hub").setLevel(logging.WARNING)
logging.getLogger("sentence_transformers").setLevel(logging.WARNING)

disable_progress_bar()

# Configure HuggingFace API key
load_dotenv()
os.environ["HF_TOKEN"] = os.getenv("HF_READ_TOKEN") if os.getenv("HF_READ_TOKEN") else "" # type: ignore

# Configure WandB info
ist_now = datetime.now(ZoneInfo("Asia/Kolkata"))
WANDB_RUN_NAME = f"{RANK_MODEL.split('/')[-1]}_{ist_now:%Y-%m-%d_%H-%M-%S}_IST"

wandb.login(key=os.getenv("WANDB_API_KEY")) if os.getenv("WANDB_API_KEY") else "" # type: ignore

In [ ]:
# Define function to create dataset for fine-tuning with CrossEncoderTrainer
def create_hf_dataset(df: pl.DataFrame, option_cols: list[str] = ["A", "B", "C", "D", "E"]) -> Dataset:
    df = df.with_columns([
        pl.concat_list([
            pl.col(opt).fill_null(" ").cast(pl.String) for opt in option_cols
        ]).alias("docs"),
        
        pl.concat_list([
            pl.when(pl.col("answer") == pl.lit(opt)).then(1.0).otherwise(0.0) for opt in option_cols
        ]).alias("scores")
    ])
    
    data_dict = df.select([
        pl.col("prompt").alias("query"), 
        "docs", 
        "scores"
    ]).to_dict(as_series=False)
    
    return Dataset.from_dict(data_dict)

In [ ]:
# Define function to compute Mean Average Precision@3 (MAP@3) [compute_map3]
def compute_map3(scores: np.ndarray, df: pl.DataFrame) -> float:
    scores = np.asarray(scores).reshape(len(df), 5)
    true = df["label_id"].to_numpy()

    top3 = np.argsort(-scores, axis=1)[:, :3]
    hit = top3 == true[:, None]
    ranks = np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, 0)

    out = np.zeros_like(ranks, dtype=float)
    np.divide(1.0, ranks, out=out, where=ranks > 0)
    
    return float(out.mean())

In [ ]:
# Get embeddings for dimensionality reduction and clustering
embed_model = SentenceTransformer(EMBED_MODEL).to(DEVICE)
embeddings = embed_model.encode(
    train_data["mcq_query"].to_list(), 
    show_progress_bar=True,
    batch_size=16
)

del embed_model
reset_gpu()

In [ ]:
# Reduce high-dimensional embeddings to 10 dimensions using UMAP for clustering
umap_model = UMAP(
    n_components=10,
    n_neighbors=30,
    min_dist=0.0,
    metric="cosine",
    n_jobs=1,
    random_state=SEED
)

umap_data = umap_model.fit_transform(embeddings)

disconnected = disconnected_vertices(umap_model)
valid_indices = np.where(~disconnected)[0]

umap_data = umap_data[valid_indices] # type: ignore
valid_ids = np.array(ids)[valid_indices]

# Find clusters from 10 dimensional embeddings for better cross-validation
clusterer_umap = HDBSCAN(
    min_cluster_size=7,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels_umap = clusterer_umap.fit_predict(umap_data)

valid_clusters = np.array(cluster_labels_umap)

# Check cluster information
n_clusters = len(set(valid_clusters))
print("Number of clusters:", n_clusters)

# cluster_sizes = (
#     pl.DataFrame({"cluster": valid_clusters})
#     .group_by("cluster")
#     .len()
#     .sort("len", descending=True)
# )
# print(cluster_sizes)

# Add cluster information to train datafrane
cluster_data = pl.DataFrame({
    "id": valid_ids,
    "cluster": valid_clusters
})

train_data = train_data.join(
    cluster_data.select("id", "cluster"),
    on="id",
    how="left"
) 

In [ ]:
# # Get baseline scores for pre trained rerankers
# gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED) # type: ignore

# X = np.zeros(len(train_data))
# groups = train_data["cluster"].fill_null(-1).to_numpy()

# fold_scores = []

# model = CrossEncoder(
#     RANK_MODEL, 
#     num_labels=1,
#     trust_remote_code=True,
#     model_kwargs={
#         "quantization_config": bnb_config,
#         "dtype": torch.float16,
#         "device_map": "cuda:0"
#     },
#     prompts={"rerank": MCQ_RERANK_INSTRUCTION},
#     default_prompt_name="rerank"
# )

# # Start iterating through {N_SPLITS}
# for fold, (train_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):

#     fold_start = time.perf_counter()
#     print(f"FOLD {fold+1}/{N_SPLITS} | "
#           f"Train Set Size: {len(train_idx)} | Val Set Size: {len(val_idx)}")

#     train_fold = train_data[train_idx]
#     val_fold = train_data[val_idx]
    
#     pairs = []
#     for row in val_fold.iter_rows(named=True):
#         query = row["prompt"]

#         for opt in OPTION_COLS:
#             doc = str(row[opt]) if row[opt] is not None else " "
#             pairs.append((query, doc))
            
#     scores = model.predict(pairs, batch_size=32, show_progress_bar=True)
    
#     fold_map3 = compute_map3(scores, val_fold) # type: ignore
#     fold_scores.append(fold_map3)
#     reset_gpu()

#     print(f"Fold {fold + 1} MAP@3: {fold_map3:.8f}\n")

# reset_gpu()

# print(f"Overall Zero-Shot Baseline MAP@3: {np.mean(fold_scores):.8f}")

In [ ]:
# # Run inference on test set with baseline pretrained models
# test_pairs = []
# for row in test_data.iter_rows(named=True):
#     query = row["prompt"]

#     for opt in OPTION_COLS:
#         doc = str(row[opt]) if row[opt] is not None else " "
#         test_pairs.append((query, doc))

# # Send test pairs to model to rank options
# test_scores = model.predict(test_pairs, batch_size=32, show_progress_bar=True)

# test_scores = np.asarray(test_scores).reshape(len(test_data), 5)

# # Extract the top 3 predictions 
# top3_indices = np.argsort(-test_scores, axis=1)[:, :3]

# predictions = []
# for indices in top3_indices:
#     pred_string = " ".join([ID2LABEL[idx] for idx in indices])
#     predictions.append(pred_string)

# del model
# reset_gpu()

In [ ]:
# # Build submission csv
# submission_data = pl.DataFrame({
#     "ID": test_data["id"],
#     "Prediction": predictions
# })

# submission_data.write_csv("submission.csv")
# print("Submission saved to submission.csv successfully.")

# print(submission_data.sample(10, seed=SEED))

- Qwen Overall Zero-Shot Baseline MAP@3: 0.75253585 [0.59642]
- Zeroentropy Overall Zero-Shot Baseline MAP@3: 0.82735766 [0.63881]

In [ ]:
# Define callback class to save best state evaluated by validation MAP@3
class BestAdapterCallback(TrainerCallback):
    def __init__(self, val_fold: pl.DataFrame, option_cols: list[str]):
        self.val_fold = val_fold
        self.option_cols = option_cols
        self.best_map3 = -1.0
        self.best_state = None

    def on_evaluate(self, args, state, control, model=None, **kwargs):
        val_pairs = []
        for row in self.val_fold.iter_rows(named=True):
            query = row["prompt"]

            for opt in self.option_cols:
                doc = str(row[opt]) if row[opt] is not None else " "
                val_pairs.append((query, doc))

        scores = model.predict(val_pairs, batch_size=32) # type: ignore
        map3 = compute_map3(np.asarray(scores), self.val_fold)
        print(f"\tEpoch {state.epoch:.0f} | MAP@3: {map3:.6f}")

        if map3 > self.best_map3:
            self.best_map3 = map3
            self.best_state = copy.deepcopy(get_peft_model_state_dict(model.transformers_model)) # type: ignore

In [ ]:
# Fine-tune over {N_SPLITS} with LamndaLoss(k=3) using 8-bit QLoRA
fold_scores = []
oof_preds = np.zeros((len(train_data), 5), dtype=np.float32)

gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED) # type: ignore

X = np.zeros(len(train_data))
groups = train_data["cluster"].fill_null(-1).to_numpy()

# Start iterating through {N_SPLITS}
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, groups=groups)):
    fold_start = time.perf_counter()
    print(f"FOLD {fold+1}/{N_SPLITS} | "
    f"Train Set Size: {len(train_idx)} | Val Set Size: {len(val_idx)}")

    train_fold = train_data[train_idx]
    val_fold = train_data[val_idx]

    train_dataset = create_hf_dataset(train_fold)
    val_dataset = create_hf_dataset(val_fold)

    # Configure WandB logging
    wandb.init(
        entity="24f2005537-dl-genai-project",
        project="dl-genai-project",
        name=f"{WANDB_RUN_NAME}-fold-{fold+1}",
        group=WANDB_RUN_NAME,
        config={
            "fold": fold + 1,
            "model": {
                "name": RANK_MODEL
            },
            "training": {
                "splits": N_SPLITS, 
                "epochs": EPOCHS, 
                "loss": "LambdaLoss"
            },
            "data": {
                "batch_size": PER_DEVICE_BATCH_SIZE, 
                "grad_accumulation_steps": GRAD_ACCUMULATION_STEPS, 
                "embed_model": EMBED_MODEL
            }
        }
    )

    # Reinitialize Base Model Configuration
    model = CrossEncoder(
        RANK_MODEL,
        num_labels=1,
        trust_remote_code=True,
        model_kwargs={
            "quantization_config": qlora_bnb_config,
            "device_map": "cuda:0"
        },
        prompts={"rerank": MCQ_RERANK_INSTRUCTION},
        default_prompt_name="rerank"
    )

    # Inject Trainable LoRA Adapters
    prepare_model_for_kbit_training(model.transformers_model)

    peft_config = LoraConfig(
        task_type=TaskType.FEATURE_EXTRACTION,
        r=8,
        lora_alpha=16,
        # r=16,
        # lora_alpha=32,
        lora_dropout=0.1,
        bias="none",
        # target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    )

    model.add_adapter(peft_config) # type: ignore
    loss = LambdaLoss(model, k=3)

    # Configure Modified Training Arguments
    training_args = CrossEncoderTrainingArguments(
        output_dir=f"./qlora-fold-{fold}",
        per_device_train_batch_size=PER_DEVICE_BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUMULATION_STEPS,

        gradient_checkpointing_kwargs={"use_reentrant": False},

        learning_rate=2e-4,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_steps=0.1,
        bf16=True,

        logging_steps=10,
        eval_strategy="epoch",

        save_strategy="no",
        report_to="wandb"
    )

    best_adapter_cb = BestAdapterCallback(val_fold, OPTION_COLS)

    # Execute Trainer
    trainer = CrossEncoderTrainer(
        model=model,
        args=training_args,

        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        loss=loss,

        callbacks=[best_adapter_cb]
    )

    trainer.train()

    if best_adapter_cb.best_state is not None:
        set_peft_model_state_dict(model.transformers_model, best_adapter_cb.best_state)

    best_score = best_adapter_cb.best_map3
    fold_scores.append(best_score)

    fold_time = time.perf_counter() - fold_start
    print(
    f"\nFold {fold+1} completed in {int(fold_time // 60)} min {fold_time % 60:.2f}s"
    f" | Best QLoRA MAP@3: {best_score:.6f}\n"
    )

    val_pairs = []
    for row in val_fold.iter_rows(named=True):
        query = row["prompt"]

        for opt in OPTION_COLS:
            doc = str(row[opt]) if row[opt] is not None else " "
            val_pairs.append((query, doc))

    # Save best fold state by MAP@3 
    val_scores = model.predict(val_pairs, batch_size=32)
    oof_preds[val_idx] = np.asarray(val_scores).reshape(-1, 5)

    torch.save(best_adapter_cb.best_state, f"./qlora-mcq-reranker-best-fold-{fold}.pt")

    # wandb.finish()
    del model, trainer, loss
    reset_gpu()

print(f"\nFinal Cross-Validation MAP@3 (Averaged across folds): {np.mean(fold_scores):.8f}") 